## Preparing the dataset for RF (NOT READY YET)

Best results (pearson correlation) for window size: variable/size/p correlation

NDVI class 1      /  15  /  -0.72

NDVI class 0.5    /  6   /   0.02

NDWI class 1      /  8   /  -0.1

Volumen veg       /  7   /  -0.61

volumen edif      /  9   /   0.42

area edif         /  8   /   0.50

elev mean         /  1   /  -0.15

slope mean        /  100 /  -0.65

northness         /  60  /  -0.28

eastness          /  11  /  -0.17

In [ ]:
import os
import numpy as np
import rasterio
from rasterio import mask
import geopandas as gpd
import matplotlib.pyplot as plt


In [ ]:
# variables
lst_clipped_path = '../../datos-notebooks/0.LST/LST_clipped_celsius.tif'
ndvi_1 = '../../datos-notebooks/1.land-cover/moving-windows/ndvi/Class_1_window15.tif'
ndwi_1 = '../../datos-notebooks/1.land-cover/moving-windows/ndwi/Class_1_window8.tif'
veg_vol = '../../datos-notebooks/2.objects-height/moving-windows/tree-volume/trees_window7.tif'
built_vol = '../../datos-notebooks/2.objects-height/moving-windows/building-volume/buildings_window9.tif'
built_area = '../../datos-notebooks/2.objects-height/moving-windows/building-area/building_area_window8.tif'
mean_elev = '../../datos-notebooks/3.topography/dtm_30m/elev_mean/elev_mean_win1.tif'
mean_slope = '../../datos-notebooks/3.topography/dtm_30m/slope_mean/slope_mean_win100.tif'
northness = '../../datos-notebooks/3.topography/dtm_30m/northness/northness_win60.tif'

output_dir = '../../datos-notebooks/4.LST-prediction/'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Diccionario de variables
rasters = {
    "LST": lst_clipped_path,
    "NDVI": ndvi_1,
    "NDWI": ndwi_1,
    "Vegetation Volume": veg_vol,
    "Building Volume": built_vol,
    "Building Area": built_area,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}

n_vars = len(rasters)
cols = 3
rows = (n_vars + cols - 1) // cols  # ceil division

fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))

for ax, (name, path) in zip(axes.flat, rasters.items()):
    with rasterio.open(path) as src:
        data = src.read(1)
        img = ax.imshow(data, cmap='viridis')
        ax.set_title(name)
        ax.axis('off')
        fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)

# Hide any unused subplots
for ax in axes.flat[n_vars:]:
    ax.axis('off')

plt.tight_layout()
plt.show()


### stratified sampling of LST

In [ ]:
import rasterio
import numpy as np
import pandas as pd

# Dictionary of rasters
rasters = {
    "LST": lst_clipped_path,
    "NDVI": ndvi_1,
    "NDWI": ndwi_1,
    "VegetationVolume": veg_vol,
    "BuildingVolume": built_vol,
    "BuildingArea": built_area,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}

# Read and flatten
data_arrays = {}
for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        data_arrays[name] = arr.flatten()

df = pd.DataFrame(data_arrays)

# Drop rows with NaN
df = df.dropna()

# Create LST bins for stratification (5 bins)
n_bins = 5
df['LST_bin'] = pd.qcut(df['LST'], n_bins, labels=False)

# Stratified sampling: 5% of each bin
sample_fraction = 0.05
df_sampled = df.groupby('LST_bin', group_keys=False).apply(
    lambda x: x.sample(frac=sample_fraction, random_state=42)
)

# Drop bin column
df_sampled = df_sampled.drop(columns=['LST_bin']).reset_index(drop=True)

print(df_sampled.head())
print("Sampled dataset shape:", df_sampled.shape)

# Split features and target
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']


In [ ]:
# Number of rows and columns
print(df_sampled.shape)  
# Output: (number_of_pixels, number_of_variables)

# Number of pixels (rows)
print(len(df_sampled))

# Number of features (columns) excluding target
print(X.shape[1])

# Quick info about dataset
print(df_sampled.info())


In [ ]:
import pandas as pd

# Recreate bins for the sampled dataset
n_bins = 5
df_sampled['LST_bin'] = pd.qcut(df_sampled['LST'], n_bins, labels=False)

# Show min and max LST in each bin
for b in range(n_bins):
    values_in_bin = df_sampled[df_sampled['LST_bin'] == b]['LST']
    print(f"Bin {b}: min = {values_in_bin.min():.2f}, max = {values_in_bin.max():.2f}, count = {len(values_in_bin)}")



In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlation matrix (only features)
corr_matrix = X.corr()

# Display heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()
